Subtasks
- Create the catalog and schemas — Use CREATE CATALOG / CREATE SCHEMA. Hint: add a COMMENT to each while you're here; LAB-15 depends on it.
- Create the volume — Hint: managed volumes need no external location. Path shape is /Volumes/<catalog>/<schema>/<volume>/. See volumes.
- Create a group and add yourself — Workspace settings → Identity and access. Hint: Free Edition has no account console; workspace admins can add users but not remove them.
- Grant SELECT at catalog level only — Then, as a thought experiment, list what else the group needs.
- Verify the inheritance chain — Hint: the missing pieces are USE CATALOG and USE SCHEMA. Inheritance is not traversal. This is exam question shaped.
- Record the grant state — Run SHOW GRANTS ON CATALOG ... and keep the output.

## Changelog 
**[0.0.1] - 22/08/2026 10PM**
### Subtask Create the catalog and schemas
- Catalog and schemas have been created!  

**[0.0.2] - 26/08/2026 10PM**
### Finish rest of the subtasks
- Create Volume
- Create group, add user to group
- Add SELECT CATALOG for group
- Verify inheritance chain
- Record the GRANTS state

In [0]:
%sql
-- Create dev_ppetkov1 Catalog

CREATE CATALOG IF NOT EXISTS dev_ppetkov1
COMMENT 'Personal Development catalog for DE Professional Certification Labs
Owner: pavlin.petkov420
Layout: bronze | silver | gold | ops
Non-production safe to drop and rebuild';


In [0]:
%sql
-- Crete bronze schema

CREATE SCHEMA IF NOT EXISTS dev_ppetkov1.bronze
COMMENT 'Bronze layer for raw data ingestion. No quality expectations. Records failing expectations are dropped silently. Rebuildable from source'
WITH DBPROPERTIES (
    'layer'            = 'bronze',
    'data_quality'     = 'none',
    'write_pattern'    = 'append-only',
    'rebuildable'      = 'true'
);

In [0]:
%sql
-- Crete silver schema

CREATE SCHEMA IF NOT EXISTS dev_ppetkov1.silver
COMMENT 'Cleansed, conformed, deduplicated data. Types cast and enforced; quality expectations applied on write. Records failing expectations are routed to ops.quarantine rather than dropped silently. Business logic and aggregations belong in gold, not here. Rebuildable from bronze.'
WITH DBPROPERTIES (
    'layer'            = 'silver',
    'data_quality'     = 'expectations_enforced',
    'write_pattern'    = 'append_and_upsert',
    'rebuildable'      = 'true',
    'upstream'         = 'bronze',
    'quarantine_target'= 'ops.quarantine'
);

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS dev_ppetkov1.gold
COMMENT 'Curated, consumer-facing datasets: business aggregates, dimensional models, and reporting tables. Assumes silver has already enforced quality — no cleansing here. Dimensions maintained as SCD Type 2 where history matters; layout optimised with liquid clustering on actual filter columns. Rebuildable from silver.'
RETAIN DROPPED FOR 30 DAYS
WITH DBPROPERTIES (
    'layer'            = 'gold',
    'data_quality'     = 'curated',
    'write_pattern'    = 'materialized_view_and_scd2',
    'rebuildable'      = 'true',
    'upstream'         = 'silver',
    'consumers'        = 'bi_analysts'
);

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS dev_ppetkov1.ops
COMMENT 'Operational and control assets — not business data. Contains the landing volume for raw files, quarantined records rejected by silver expectations, data quality metrics, and pipeline control tables. NOT rebuildable: quarantine and metrics are the only record of what was rejected and when. Treat as append-only evidence.'
RETAIN DROPPED FOR 30 DAYS
WITH DBPROPERTIES (
    'layer'            = 'ops',
    'data_quality'     = 'n/a',
    'write_pattern'    = 'append_only_evidence',
    'rebuildable'      = 'false',
    'upstream'         = 'silver_rejects',
    'contains_pii'     = 'true'
);

Date: 2026-08-26

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS dev_ppetkov1.ops.landing
COMMENT 'Raw file landing zone for Auto Loader → bronze. Populated by the LAB-02 export from samples.nyctaxi.trips. Contains intentional schema-drift and dirty-record files for LAB-03/LAB-06 — these are test fixtures, not errors. Reconstruction source for bronze.';

In [0]:
%sql
select 1
-- Statements below creates a legacy workspace group, which cannot be granted select access on catalog only for example
-- This is important and being tested during exams
/*-- This group was already created via UI
CREATE GROUP dev_ppetkov1_group_2 
WITH USER 'pavlin.petkov420@protonmail.com';*/

In [0]:
%sql
GRANT SELECT ON CATALOG dev_ppetkov1 to dev_ppetkov1_group;

In [0]:
%sql
SHOW GRANTS ON CATALOG dev_ppetkov1;

# Check access/identity chain

In [0]:
%sql
SHOW GRANTS ON SCHEMA dev_ppetkov1.ops;

In [0]:
%sql
SHOW TABLES IN dev_ppetkov1.information_schema;

In [0]:
%sql
SELECT current_user(), is_account_group_member('dev_ppetkov1_group') as is_member_of_dev_ppetkov_1_group;